# SGD with momentum over a nested pytree

JAX parameters are **pytrees**: nested dicts, lists and tuples with arrays at the leaves. An optimizer is a function that maps `(params, state, grads)` to new values with the same structure, usually written with `jax.tree_util.tree_map` (or `jax.tree.map`).

Write `momentum_step(params, velocity, grads, lr, beta, weight_decay)` using the PyTorch convention for every leaf $p$ with gradient $g$ and velocity $v$:

$$g' = g + \lambda\, p, \qquad v \leftarrow \beta\, v + g', \qquad p \leftarrow p - \eta\, v$$

- `params`, `grads`: pytrees with the same structure (nested dicts of arrays of any shape, including 0-d).
- `velocity`: a pytree of the same structure, or `None` on the very first step, meaning all zeros.
- `lr` $=\eta$, `beta` $=\beta$, `weight_decay` $=\lambda$: floats.

Return `(new_params, new_velocity)`, both with the same structure as `params`. float32 leaves. Do not hard-code the key names: the tests use different trees.

This problem runs in **Colab** (JAX isn't available in the browser). Use the default CPU runtime; JAX computes in float32 here. The notebook includes the same tests.

Write your solution in the next cell, then run the test cell.

In [ ]:
import jax
import jax.numpy as jnp

def momentum_step(params, velocity, grads, lr, beta, weight_decay):
    """Return (new_params, new_velocity), same pytree structure as params."""
    raise NotImplementedError


In [ ]:
# Tests (same harness and cases as the website)
_HARNESS = '"""Test harness shared by build.py (CPython) and the browser runner (Pyodide).\n\nOne file, so a test that passes at build time is graded the same way in the browser.\nValues cross JSON with small tags: ndarray, tuple, DataFrame/Series, non-finite floats.\n"""\nimport json, math\n\ntry:\n    import numpy as np\nexcept ImportError:  # pure-python problems never need it\n    np = None\n\n\ndef enc(x):\n    if np is not None and isinstance(x, np.ndarray):\n        return {\'__nd__\': enc(x.tolist()), \'dtype\': str(x.dtype)}\n    if np is not None and isinstance(x, np.generic):\n        return enc(x.item())\n    if isinstance(x, float) and not math.isfinite(x):\n        return {\'__f__\': repr(x)}\n    if isinstance(x, tuple):\n        return {\'__tup__\': [enc(v) for v in x]}\n    if isinstance(x, list):\n        return [enc(v) for v in x]\n    if isinstance(x, dict):\n        return {\'__dict__\': [[enc(k), enc(v)] for k, v in x.items()]}\n    t = type(x).__name__\n    if t == \'Tensor\':  # torch, in Colab and at build time\n        return enc(x.detach().cpu().numpy())\n    if t == \'DataFrame\':\n        # Cells go through enc() as Python values, so floats keep full precision (to_json caps at 15 digits).\n        return {\'__df__\': {\'columns\': [enc(c) for c in x.columns], \'data\': [enc(list(r)) for r in x.itertuples(index=False, name=None)]},\n                \'dtypes\': [str(d) for d in x.dtypes]}\n    if t == \'Series\':\n        return {\'__sr__\': enc(x.tolist()), \'name\': x.name}\n    return x\n\n\ndef dec(x):\n    if isinstance(x, list):\n        return [dec(v) for v in x]\n    if not isinstance(x, dict):\n        return x\n    if \'__nd__\' in x:\n        return np.array(dec(x[\'__nd__\']), dtype=x[\'dtype\'])\n    if \'__f__\' in x:\n        return float(x[\'__f__\'])\n    if \'__tup__\' in x:\n        return tuple(dec(v) for v in x[\'__tup__\'])\n    if \'__dict__\' in x:\n        return {_hashable(dec(k)): dec(v) for k, v in x[\'__dict__\']}\n    if \'__df__\' in x:\n        import pandas as pd\n        s = x[\'__df__\']\n        df = pd.DataFrame([dec(r) for r in s[\'data\']], columns=[dec(c) for c in s[\'columns\']])\n        for c, d in zip(df.columns, x[\'dtypes\']):\n            if d.startswith((\'int\', \'float\', \'bool\')):\n                df[c] = df[c].astype(d)\n        return df\n    if \'__sr__\' in x:\n        import pandas as pd\n        return pd.Series(dec(x[\'__sr__\']), name=x[\'name\'])\n    return x\n\n\ndef _hashable(k):\n    return tuple(k) if isinstance(k, list) else k\n\n\ndef same(got, exp, rtol=1e-6, atol=1e-8, path=\'output\'):\n    """(ok, why). Floats compare with tolerance, everything else exactly. Shapes must match."""\n    if type(exp).__name__ == \'DataFrame\':\n        if type(got).__name__ != \'DataFrame\':\n            return False, f\'{path}: expected a DataFrame, got {type(got).__name__}\'\n        if list(got.columns) != list(exp.columns):\n            return False, f\'{path}: columns {list(got.columns)} != expected {list(exp.columns)}\'\n        if got.shape != exp.shape:\n            return False, f\'{path}: shape {got.shape} != expected {exp.shape}\'\n        for c in exp.columns:\n            ok, why = same(got[c].tolist(), exp[c].tolist(), rtol, atol, f\'{path}[{c!r}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if type(exp).__name__ == \'Series\':\n        exp = exp.tolist()\n        got = got.tolist() if type(got).__name__ == \'Series\' else got\n    if np is not None and isinstance(exp, np.ndarray):\n        try:\n            g = np.asarray(got)\n        except Exception:\n            return False, f\'{path}: expected an array, got {type(got).__name__}\'\n        if g.shape != exp.shape:\n            return False, f\'{path}: shape {g.shape} != expected {exp.shape}\'\n        if exp.dtype.kind in \'fc\' or g.dtype.kind in \'fc\':\n            if g.dtype.kind not in \'fciub\':\n                return False, f\'{path}: expected numbers, got dtype {g.dtype}\'\n            bad = ~np.isclose(g.astype(complex if exp.dtype.kind == \'c\' else float), exp, rtol=rtol, atol=atol, equal_nan=True)\n            if bad.any():\n                i = tuple(int(v) for v in np.argwhere(bad)[0])\n                return False, f\'{path}{list(i)}: {g[i].tolist()!r} != expected {exp[i].tolist()!r}\'\n            return True, \'\'\n        if not np.array_equal(g, exp):\n            i = tuple(int(v) for v in np.argwhere(g != exp)[0]) if g.shape else ()\n            return False, f\'{path}{list(i)}: {g[i].tolist()!r} != expected {exp[i].tolist()!r}\'\n        return True, \'\'\n    if isinstance(exp, float) or (isinstance(exp, int) and isinstance(got, float) and not isinstance(exp, bool)):\n        if np is not None and isinstance(got, np.generic):\n            got = got.item()\n        if not isinstance(got, (int, float)) or isinstance(got, bool):\n            return False, f\'{path}: expected a number, got {type(got).__name__}\'\n        if math.isnan(exp) and math.isnan(got):\n            return True, \'\'\n        return (True, \'\') if math.isclose(got, exp, rel_tol=rtol, abs_tol=atol) else (False, f\'{path}: {got!r} != expected {exp!r}\')\n    if isinstance(exp, (list, tuple)):\n        if np is not None and isinstance(got, np.ndarray):\n            got = got.tolist()\n        if not isinstance(got, (list, tuple)):\n            return False, f\'{path}: expected a {type(exp).__name__}, got {type(got).__name__}\'\n        if len(got) != len(exp):\n            return False, f\'{path}: length {len(got)} != expected {len(exp)}\'\n        for i, (g, e) in enumerate(zip(got, exp)):\n            ok, why = same(g, e, rtol, atol, f\'{path}[{i}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if isinstance(exp, dict):\n        if not isinstance(got, dict):\n            return False, f\'{path}: expected a dict, got {type(got).__name__}\'\n        if set(got) != set(exp):\n            return False, f\'{path}: keys {sorted(map(str, got))} != expected {sorted(map(str, exp))}\'\n        for k in exp:\n            ok, why = same(got[k], exp[k], rtol, atol, f\'{path}[{k!r}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if np is not None and isinstance(got, np.generic):\n        got = got.item()\n    return (True, \'\') if got == exp else (False, f\'{path}: {got!r} != expected {exp!r}\')\n\n\ndef run_sql(query, setup):\n    """Run `setup` (CREATE/INSERT script) in a fresh in-memory db, then `query`; rows as lists."""\n    import sqlite3  # a separate package in Pyodide: only SQL problems load it\n    con = sqlite3.connect(\':memory:\')\n    try:\n        con.executescript(setup)\n        cur = con.execute(query)\n        return [[c[0] for c in cur.description], [list(r) for r in cur.fetchall()]]\n    finally:\n        con.close()\n\n\ndef show_args(args, runtime, n=600):\n    """Arguments as the learner would write them: one per line, SQL setup as-is."""\n    if runtime == \'sql\':\n        return args[0] if len(args[0]) <= n else args[0][:n] + \' …\'\n    return \'\\n\'.join(short(a, n) for a in args)\n\n\ndef short(x, n=300):\n    s = repr(x)\n    return s if len(s) <= n else s[:n] + \' …\'\n\n\ndef grade(fn, tests, runtime, rtol=1e-6, atol=1e-8):\n    """Run `fn` (a callable, or SQL text for runtime \'sql\') on each test; list of result dicts."""\n    out = []\n    for t in tests:\n        args, exp = prep(dec(t[\'a\']), runtime), dec(t[\'e\'])\n        try:\n            got = run_sql(fn, args[0]) if runtime == \'sql\' else plain(fn(*_copy(args)))\n            ok, why = same(got, exp, rtol, atol)\n        except Exception as e:  # learner code raised: report it like a failed test\n            got, ok, why = None, False, f\'{type(e).__name__}: {e}\'\n        out.append({\'ok\': ok, \'why\': why, \'got\': short(got) if not ok else \'\',\n                    \'inp\': show_args(args, runtime, 400) if not ok else \'\'})\n    return out\n\n\ndef prep(args, runtime):\n    """Arrays cross JSON as ndarrays; torch problems take tensors."""\n    if runtime == \'torch\':\n        import torch\n        return [torch.as_tensor(a) if np is not None and isinstance(a, np.ndarray) else a for a in args]\n    if runtime == \'triton\':                                  # Colab only: kernels need CUDA tensors\n        import torch\n        return [torch.as_tensor(a).cuda() if np is not None and isinstance(a, np.ndarray) else a for a in args]\n    return args\n\n\ndef plain(x):\n    """Framework arrays (torch, jax) -> numpy, recursively, so results compare and encode the same way."""\n    t = type(x).__name__\n    if t == \'Tensor\':\n        return x.detach().cpu().numpy()\n    if t in (\'ArrayImpl\', \'Array\') and hasattr(x, \'__array__\'):\n        return np.asarray(x)\n    if isinstance(x, tuple):\n        return tuple(plain(v) for v in x)\n    if isinstance(x, list):\n        return [plain(v) for v in x]\n    if isinstance(x, dict):\n        return {k: plain(v) for k, v in x.items()}\n    return x\n\n\ndef _copy(args):\n    # Tests reuse inputs across Run and Submit, so a function that mutates its input mustn\'t poison later cases.\n    import copy\n    return copy.deepcopy(args)\n'
_TESTS = '[{"a": [{"__dict__": [["w", {"__nd__": [1.0, 2.0], "dtype": "float64"}]]}, null, {"__dict__": [["w", {"__nd__": [0.5, -1.0], "dtype": "float64"}]]}, 0.1, 0.9, 0.0], "e": {"__tup__": [{"__dict__": [["w", {"__nd__": [0.949999988079071, 2.0999999046325684], "dtype": "float32"}]]}, {"__dict__": [["w", {"__nd__": [0.5, -1.0], "dtype": "float32"}]]}]}}, {"a": [{"__dict__": [["w", {"__nd__": [1.0, 2.0], "dtype": "float64"}]]}, {"__dict__": [["w", {"__nd__": [1.0, 1.0], "dtype": "float64"}]]}, {"__dict__": [["w", {"__nd__": [0.0, 0.0], "dtype": "float64"}]]}, 0.1, 0.9, 0.0], "e": {"__tup__": [{"__dict__": [["w", {"__nd__": [0.9100000262260437, 1.909999966621399], "dtype": "float32"}]]}, {"__dict__": [["w", {"__nd__": [0.8999999761581421, 0.8999999761581421], "dtype": "float32"}]]}]}}, {"a": [{"__dict__": [["dense", {"__dict__": [["w", {"__nd__": [[-0.5947236978403764, 0.630783487419836], [1.0393541232237575, 1.0309217445385483], [1.8178460987289433, -0.3851893808502385]], "dtype": "float64"}], ["b", {"__nd__": [0.5441771887526803, -0.36622168185723664], "dtype": "float64"}]]}], ["scale", {"__nd__": -1.4248485130063353, "dtype": "float64"}]]}, null, {"__dict__": [["dense", {"__dict__": [["w", {"__nd__": [[-0.7038591205388717, 0.1361623384532617], [-0.9151747291399203, -0.19147101971894068], [1.1202500757479104, 0.5704516907585605]], "dtype": "float64"}], ["b", {"__nd__": [0.5723436551805909, 0.34512242791942843], "dtype": "float64"}]]}], ["scale", {"__nd__": -0.1752757919726442, "dtype": "float64"}]]}, 0.01, 0.9, 0.01], "e": {"__tup__": [{"__dict__": [["dense", {"__dict__": [["b", {"__nd__": [0.538399338722229, -0.36963629722595215], "dtype": "float32"}], ["w", {"__nd__": [[-0.5876256227493286, 0.6293587684631348], [1.048401951789856, 1.0327333211898804], [1.8064618110656738, -0.3908553719520569]], "dtype": "float32"}]]}], ["scale", {"__nd__": -1.4229533672332764, "dtype": "float32"}]]}, {"__dict__": [["dense", {"__dict__": [["b", {"__nd__": [0.5777854323387146, 0.3414601981639862], "dtype": "float32"}], ["w", {"__nd__": [[-0.7098063230514526, 0.14247018098831177], [-0.9047811627388, -0.1811618059873581], [1.1384285688400269, 0.5665997862815857]], "dtype": "float32"}]]}], ["scale", {"__nd__": -0.18952427804470062, "dtype": "float32"}]]}]}}, {"a": [{"__dict__": [["dense", {"__dict__": [["w", {"__nd__": [[-1.867742907984139, 0.9914585596358675], [-1.5066600744223568, 0.21608504316221636], [-0.10839172496544125, 0.1372724260249292]], "dtype": "float64"}], ["b", {"__nd__": [0.25233676853552633, -0.33384257248018917], "dtype": "float64"}]]}], ["scale", {"__nd__": 0.9009835234430289, "dtype": "float64"}]]}, {"__dict__": [["dense", {"__dict__": [["w", {"__nd__": [[-1.2851013197016634, 0.7919045814471145], [-1.6919469283692645, 1.1863323624397912], [-0.5089677788430257, 0.3740524211260804]], "dtype": "float64"}], ["b", {"__nd__": [1.5080552589573382, -2.162682426982347], "dtype": "float64"}]]}], ["scale", {"__nd__": -0.31462817801197823, "dtype": "float64"}]]}, {"__dict__": [["dense", {"__dict__": [["w", {"__nd__": [[0.573498241215757, 0.8992781221991692], [1.3582785284075283, 0.6061457288054009], [0.5141360357068666, 0.1063379638618435]], "dtype": "float64"}], ["b", {"__nd__": [0.26556570443659167, 0.07098076750143006], "dtype": "float64"}]]}], ["scale", {"__nd__": 0.9918099909518594, "dtype": "float64"}]]}, 0.05, 0.5, 0.0], "e": {"__tup__": [{"__dict__": [["dense", {"__dict__": [["b", {"__nd__": [0.20135711133480072, -0.28332453966140747], "dtype": "float32"}], ["w", {"__nd__": [[-1.8642902374267578, 0.9266970157623291], [-1.5322753190994263, 0.1561194509267807], [-0.12137433141469955, 0.12260422110557556]], "dtype": "float32"}]]}], ["scale", {"__nd__": 0.8592587113380432, "dtype": "float32"}]]}, {"__dict__": [["dense", {"__dict__": [["b", {"__nd__": [1.0195932388305664, -1.0103604793548584], "dtype": "float32"}], ["w", {"__nd__": [[-0.06905239820480347, 1.2952303886413574], [0.5123050212860107, 1.1993119716644287], [0.25965213775634766, 0.29336416721343994]], "dtype": "float32"}]]}], ["scale", {"__nd__": 0.8344959020614624, "dtype": "float32"}]]}]}}, {"a": [{"__dict__": [["a", {"__dict__": [["b", {"__dict__": [["c", {"__nd__": 3.0, "dtype": "float64"}]]}]]}], ["d", {"__nd__": [1.0, 1.0, 1.0, 1.0], "dtype": "float64"}]]}, {"__dict__": [["a", {"__dict__": [["b", {"__dict__": [["c", {"__nd__": 1.0, "dtype": "float64"}]]}]]}], ["d", {"__nd__": [0.0, 0.0, 0.0, 0.0], "dtype": "float64"}]]}, {"__dict__": [["a", {"__dict__": [["b", {"__dict__": [["c", {"__nd__": -2.0, "dtype": "float64"}]]}]]}], ["d", {"__nd__": [0.5, 0.5, 0.5, 0.5], "dtype": "float64"}]]}, 0.1, 0.0, 0.1], "e": {"__tup__": [{"__dict__": [["a", {"__dict__": [["b", {"__dict__": [["c", {"__nd__": 3.1700000762939453, "dtype": "float32"}]]}]]}], ["d", {"__nd__": [0.9399999976158142, 0.9399999976158142, 0.9399999976158142, 0.9399999976158142], "dtype": "float32"}]]}, {"__dict__": [["a", {"__dict__": [["b", {"__dict__": [["c", {"__nd__": -1.7000000476837158, "dtype": "float32"}]]}]]}], ["d", {"__nd__": [0.6000000238418579, 0.6000000238418579, 0.6000000238418579, 0.6000000238418579], "dtype": "float32"}]]}]}}, {"a": [{"__dict__": [["dense", {"__dict__": [["w", {"__nd__": [[0.15933383800886564, 0.6688719814697719], [1.616138731584835, 1.1701846660515787], [-0.34412485596041004, -0.10342659228173565]], "dtype": "float64"}], ["b", {"__nd__": [0.7348236865569762, -0.6423035005046784], "dtype": "float64"}]]}], ["scale", {"__nd__": -1.8309186111419358, "dtype": "float64"}]]}, {"__dict__": [["dense", {"__dict__": [["w", {"__nd__": [[-1.0114345080720917, 0.5386567206127174], [-0.21106057365786446, -0.3569138003697271], [0.7470687888737193, -1.0516975093361867]], "dtype": "float64"}], ["b", {"__nd__": [2.2645049256275085, 0.0230375564818651], "dtype": "float64"}]]}], ["scale", {"__nd__": -0.3241790209463744, "dtype": "float64"}]]}, {"__dict__": [["dense", {"__dict__": [["w", {"__nd__": [[2.0369901103033285, -1.089018974622601], [-0.4972828238457539, -0.08944615314540644], [-0.960287912185924, -0.340113747176087]], "dtype": "float64"}], ["b", {"__nd__": [0.5826229407469505, 0.16938230109984248], "dtype": "float64"}]]}], ["scale", {"__nd__": -0.048528331810538054, "dtype": "float64"}]]}, 0.1, 0.99, 0.5], "e": {"__tup__": [{"__dict__": [["dense", {"__dict__": [["b", {"__nd__": [0.4156342148780823, -0.6294072866439819], "dtype": "float32"}], ["w", {"__nd__": [[0.04780014604330063, 0.6910032629966736], [1.6059550046920776, 1.1559544801712036], [-0.30484962463378906, 0.03987416625022888]], "dtype": "float32"}]]}], ["scale", {"__nd__": -1.7024261951446533, "dtype": "float32"}]]}, {"__dict__": [["dense", {"__dict__": [["b", {"__nd__": [3.191894769668579, -0.12896227836608887], "dtype": "float32"}], ["w", {"__nd__": [[1.1153368949890137, -0.22131282091140747], [0.10183656215667725, 0.14230147004127502], [-0.3927522599697113, -1.4330075979232788]], "dtype": "float32"}]]}], ["scale", {"__nd__": -1.2849249839782715, "dtype": "float32"}]]}]}}, {"a": [{"__dict__": [["emb", {"__nd__": [[0.9882505038165825, 1.8865714196311545, 1.5634673612356318], [-2.09335821047072, 1.3710492931711142, 0.7757242335178858], [0.40799436496000613, -0.983174386232386, -0.45195066902798575], [-0.20963631877887165, -1.3926365824435363, 1.2341779304240124], [0.8906605576581846, -0.013349239608598712, -0.9480166267677115]], "dtype": "float64"}], ["head", {"__dict__": [["w", {"__nd__": [[1.3458677531896093, 0.6409924797459927, 1.0631986853724484, -0.4185880355624374], [2.39579768307788, -0.13884913569959054, 1.954754507805973, -2.004913567480772], [0.3083009845177515, 0.08254212050111535, -2.5217153157185748, -0.645112868841113]], "dtype": "float64"}]]}]]}, null, {"__dict__": [["emb", {"__nd__": [[-1.2253655808903328, -1.4066849683257712, -0.7538057419892571], [-1.1704075537758285, 0.3255216608612287, 0.46481805680877786], [0.26864010116892645, -0.8289032024163436, -0.4822407029368291], [1.6783775322778178, -0.344711291068899, -0.02078497961925633], [-0.5877514574961317, 0.5274475777626251, 0.8853307259611265]], "dtype": "float64"}], ["head", {"__dict__": [["w", {"__nd__": [[0.7955832307936058, 0.4257802909399674, 0.7104299638256821, -0.213194199761867], [0.621158712297676, -0.9893348985477781, 0.10094954635497845, 0.8649441228487049], [-1.015466657162706, 0.7789001927369343, 0.3175634206533348, -1.2251781433964828]], "dtype": "float64"}]]}]]}, 1.0, 0.9, 0.0], "e": {"__tup__": [{"__dict__": [["emb", {"__nd__": [[2.213616132736206, 3.2932562828063965, 2.3172731399536133], [-0.9229507446289062, 1.045527696609497, 0.31090617179870605], [0.13935425877571106, -0.1542711853981018, 0.030290037393569946], [-1.8880138397216797, -1.0479252338409424, 1.2549629211425781], [1.4784120321273804, -0.5407968163490295, -1.8333473205566406]], "dtype": "float32"}], ["head", {"__dict__": [["w", {"__nd__": [[0.5502845048904419, 0.2152121663093567, 0.3527687191963196, -0.20539383590221405], [1.7746390104293823, 0.8504857420921326, 1.8538049459457397, -2.8698577880859375], [1.3237676620483398, -0.6963580846786499, -2.8392786979675293, 0.5800652503967285]], "dtype": "float32"}]]}]]}, {"__dict__": [["emb", {"__nd__": [[-1.2253656387329102, -1.4066849946975708, -0.7538057565689087], [-1.1704075336456299, 0.32552164793014526, 0.4648180603981018], [0.26864010095596313, -0.8289031982421875, -0.4822407066822052], [1.6783775091171265, -0.3447113037109375, -0.020784979686141014], [-0.5877514481544495, 0.5274475812911987, 0.8853307366371155]], "dtype": "float32"}], ["head", {"__dict__": [["w", {"__nd__": [[0.7955832481384277, 0.4257802963256836, 0.7104299664497375, -0.21319420635700226], [0.6211587190628052, -0.9893348813056946, 0.10094954818487167, 0.8649441003799438], [-1.0154666900634766, 0.7789002060890198, 0.31756341457366943, -1.2251781225204468]], "dtype": "float32"}]]}]]}]}}]'
import json, types
harness = types.ModuleType('harness'); exec(_HARNESS, harness.__dict__)
res = harness.grade(momentum_step, json.loads(_TESTS), 'jax', *[1e-05, 1e-06])
for i, r in enumerate(res): print(f"test {i+1}: {'pass' if r['ok'] else 'FAIL  ' + r['why']}")
print(f"{sum(r['ok'] for r in res)}/{len(res)} passed")